In [4]:
# ./cloud_sql_proxy -instances=coastal-epigram-162302:us-east1:wikidump=tcp:3306 &
# mysql -u root -p --host 127.0.0.1

import pymysql
import pandas as pd

db_params = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "strider",
    "db": "wiki-20190501",
    "cursorclass": pymysql.cursors.DictCursor
}

connection = pymysql.connect(**db_params)

In [5]:
q_trim = """
    DELETE
    FROM
      page
    WHERE
      page_namespace <> 0
    ;
"""

q_trim2 = """
    DELETE
    FROM
      pagelinks
    WHERE
      NOT (pl_from_namespace = 0 AND pl_namespace = 0)
    ;
"""

In [6]:
with connection.cursor() as cursor:
    cursor.execute(q_trim)
print('done')

KeyboardInterrupt: 

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(q_trim2)
print('done')

In [12]:
with connection.cursor() as cursor:
    #cursor.execute(q_pre)
    cursor.execute("""SHOW TABLES
    """)
    d = pd.DataFrame(cursor.fetchall())
print(d)

  Tables_in_wiki-20190501
0           categorylinks
1                geo_tags
2                    page
3        page_title_links
4               pagelinks
5                redirect


In [24]:
q_pre = """
RESET QUERY CACHE;
"""

q = """
SELECT
    page_title AS page_title
    , page_random
FROM
    page
WHERE
    page_namespace = 0
LIMIT 100001
;
"""

In [25]:
q = """
SELECT
    page_id
    , COALESCE(id.in_degree, 0) AS in_degree
    , COALESCE(od.out_degree, 0) AS out_degree
FROM
    (
        SELECT
            pl_from AS page_id
            , COUNT(DISTINCT(pl_title)) AS out_degree
        FROM
            pagelinks
        WHERE
            pl_from IN (151296, 37847503)
            AND pl_namespace = 0
            AND pl_from_namespace = 0
        GROUP BY pl_from
    ) AS od
    LEFT JOIN
    (
        SELECT
            COUNT(DISTINCT(pl_from)) AS in_degree
            , page_id
        FROM
            page
            JOIN pagelinks ON page_title = pl_title
        WHERE
            page_id IN (151296, 37847503)
            AND pl_namespace = 0
            AND pl_from_namespace = 0
        GROUP BY page_id
    ) AS id USING(page_id)
;
"""

In [26]:
with connection.cursor() as cursor:
    #cursor.execute(q_pre)
    cursor.execute(q)
    #n = cursor.fetchmany(100)
    #node_names = list(cursor.fetchall_unbuffered())
    d = cursor.fetchall()

In [27]:
d

[{'page_id': 151296, 'in_degree': 1118, 'out_degree': 654},
 {'page_id': 37847503, 'in_degree': 32, 'out_degree': 1472}]

In [2]:
with connection.cursor() as cursor:
    cursor.execute(q)
    node_names = cursor.fetchall()
#print(node_names)

n_byte = tuple(n['page_title'] for n in node_names)
n_str = tuple(str(n['page_title'], 'utf8') for n in node_names)

In [4]:
q = """
SELECT DISTINCT
    page_title AS page_title
    , pl_title AS out_page_title
FROM
    pagelinks
    JOIN page ON page_id=pl_from
WHERE
    pl_title IN %(nodes)s
    AND pl_namespace = 0
    AND pl_from_namespace = 0
;
"""

In [5]:
q = """
SELECT
    page_title AS page_title
    , pl_title AS out_page_title
FROM
    (
        SELECT
            gt_page_id
            , MIN(
                POW(69.1 * (gt_lat - %(lat)s), 2) +
                POW(69.1 * (%(lon)s - gt_lon) * COS(gt_lon / 57.3), 2)
            ) AS distance_sqr
        FROM
            geo_tags
        WHERE
            gt_lat BETWEEN
                %(lat)s - (18 / 69.1) AND %(lat)s + (18 / 69.1)

            AND gt_lon BETWEEN
                %(lon)s - (18 / (69.1 * COS(%(lat)s)))
                AND %(lon)s + (18 / (69.1 * COS(%(lat)s)))

        GROUP BY gt_page_id
        ORDER BY distance_sqr ASC
        LIMIT %(n)s
    ) AS nearby
    JOIN page ON page_id=gt_page_id
    JOIN pagelinks AS pl ON page_title=pl_title

WHERE
    page_namespace = 0
    AND pl_namespace = 0
    AND pl_from_namespace = 0

UNION

SELECT
    page_title AS page_title
    , pl_title AS out_page_title
FROM
    (
        SELECT
            gt_page_id
            , MIN(
                POW(69.1 * (gt_lat - %(lat)s), 2) +
                POW(69.1 * (%(lon)s - gt_lon) * COS(gt_lon / 57.3), 2)
            ) AS distance_sqr
        FROM
            geo_tags
        WHERE
            gt_lat BETWEEN
                %(lat)s - (18 / 69.1) AND %(lat)s + (18 / 69.1)

            AND gt_lon BETWEEN
                %(lon)s - (18 / (69.1 * COS(%(lat)s)))
                AND %(lon)s + (18 / (69.1 * COS(%(lat)s)))

        GROUP BY gt_page_id
        ORDER BY distance_sqr ASC
        LIMIT %(n)s
    ) AS nearby
    JOIN pagelinks AS pl ON gt_page_id=pl_from
    JOIN page ON page_id=pl_from
WHERE
    page_namespace = 0
    AND pl_namespace = 0
    AND pl_from_namespace = 0
;
"""

In [6]:
q_nearby = """
    (
        SELECT
            NULL AS page_title
            , page.page_title AS out_page_title
            , MIN(
                POW(69.1 * (gt_lat - %(lat)s), 2) +
                POW(69.1 * (%(lon)s - gt_lon) * COS(gt_lon / 57.3), 2)
            ) AS distance_sqr
        FROM
            geo_tags
            JOIN page ON page_id=gt_page_id

        WHERE
            gt_lat BETWEEN
                %(lat)s - (18 / 69.1) AND %(lat)s + (18 / 69.1)

            AND gt_lon BETWEEN
                %(lon)s - (18 / (69.1 * COS(%(lat)s)))
                AND %(lon)s + (18 / (69.1 * COS(%(lat)s)))

        GROUP BY gt_page_id
        ORDER BY distance_sqr ASC
        LIMIT %(n)s
    )
"""

"""
    SELECT
        prev{i}0.page_title
        , prev{i}0.out_page_title
    FROM
        {inputs} AS prev{i}0

    UNION
"""

q_dilate = """

    SELECT
        prev{i}1.out_page_title AS page_title
        , pl.pl_title AS out_page_title
    FROM
        {inputs} AS prev{i}1
        JOIN page AS pg ON pg.page_title=prev{i}1.out_page_title
        JOIN pagelinks AS pl ON pg.page_id=pl.pl_from

    WHERE
        page_namespace = 0
        AND pl_namespace = 0
        AND pl_from_namespace = 0

    UNION

    SELECT
        pg.page_title AS page_title
        , prev{i}2.out_page_title AS out_page_title
    FROM
        {inputs} AS prev{i}2
        JOIN pagelinks AS pl ON prev{i}2.out_page_title=pl.pl_title
        JOIN page AS pg ON pg.page_id=pl.pl_from
    WHERE
        page_namespace = 0
        AND pl_namespace = 0
        AND pl_from_namespace = 0
"""

In [7]:
q = q_dilate.format(
    i=1,
    inputs="(" + q_dilate.format(i=0, inputs=q_nearby) + ")"
)

In [8]:
#q = q_dilate.format(i=0, inputs=q_nearby)

In [9]:
#%%timeit
p = {'lat': 44.8113, 'lon': -91.4985, 'n': 5}
with connection.cursor() as cursor:
    #cursor.execute(q_pre)
    cursor.execute(q, p)
    res = cursor.fetchall()

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(res)

In [ ]:
df.shape

In [ ]:
df.head(10)